In [20]:
import os
import h5py
import numpy as np
import pandas as pd
from scipy.special import logit, expit, logsumexp

from astropy.cosmology import FlatLambdaCDM, z_at_value
import astropy.constants as constants
import astropy.units as u

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import corner

import jax
import jax.numpy as jnp
import optax
import equinox as eqx

# FlowJAX (new API)
from flowjax.flows import masked_autoregressive_flow
from flowjax.distributions import Normal

from pathlib import Path

import seaborn as sns
from tqdm import tqdm, trange

import utils as ut


import matplotlib as mpl
mpl.rcParams["text.usetex"] = False

# silence unnecessary warnings about some specific model not being available (shouldn't hurt performance according to ChatGPT)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
jax.config.update('jax_enable_x64', True)

In [2]:
filename_flow = "trained_flows/CNF_Injections/CNF.npz"
loaded_flow, loaded_config = ut.load_flow(filename_flow, ut.create_flow_from_config)

### Some simple testing

note that the for loop does some (for the final purpose) illegal normalization. This was just to test that generally the approach produces sensible results.

In [9]:
# first check that e.g. low probability for events with low mass and high distance vs. the opposite. Check that it varies correctly for different FAR

fake_parameters = {
    "m1_detector": [10, 10, 100, 100], # two low mass distant events to be tested with different FARs, two high mass low distance events tested with different FARs
    "m2_detector": [8, 8, 80, 80],
    "luminosity_distance": [2, 2, 0.1, 0.1], # this stuff is in Gpc
    "a1": [0.3, 0.3, 0.3, 0.3],
    "a2": [0.2]*4,
    "cos_inclination": [0.8] * 4,
    "sinusoidal_polarization": [np.pi/2]*4,
    "t1": [1]*4,
    "t2": [1]*4,
    "cos_phi1": [0.8]*4,
    "cos_phi2": [-0.8]*4,
    "sinusoidal_right_ascension": [1.5]*4,
    "atan_declination": [1.0]*4,
}

fake_parameters_df = pd.DataFrame(fake_parameters)

fake_FARs = {
    "far_min": [1, 1e-40, 1, 1e-40],
}

fake_FARs_df = pd.DataFrame(fake_FARs)

In [10]:
# load transformations and create Data objects
X_data = ut.Data.from_saved("trained_flows/CNF_Injections/val_dataset/X/")
y_data = ut.Data.from_saved("trained_flows/CNF_Injections/val_dataset/y/")

In [12]:
# some dummy test
loaded_flow.log_prob(fake_parameters_df.to_numpy(), condition = fake_FARs_df.to_numpy())

# these numbers do not mean anything because I did not transform them!!!

Array([-1.15753125e+41, -1.30657988e+41, -6.37932177e+95, -6.15727418e+95],      dtype=float64)

In [13]:
fake_parameters_df.to_numpy().shape

(4, 13)

In [15]:
test_arr = np.array([1,3,5])
np.tile(test_arr, (5, 1)).shape

(5, 3)

### How to actually evaluate the CNF approach

1. Read the validation injection data -> the CNF must not have seen this, otherwise it's not a good evaluation
2. For each parameter and FAR combination in the data determine $P(\mathrm{detected} | \theta)$
   1. Evaluate $p(\theta | \mathrm{FAR} < \mathrm{FAR_{threshold}}) = \int_0^\mathrm{FAR_{threshold}} p_\mathrm{CNF} (\theta | \mathrm{FAR}) p(\mathrm{FAR}) d\mathrm{FAR} = \frac{1}{N_\mathrm{threshold}} \sum_i^{N_\mathrm{threshold}} p_\mathrm{CNF}(\theta | \mathrm{FAR})$, where $N_\mathrm{total}$ is the number of total FAR values for a dataset, $\theta$  are the GW parameters and $N_\mathrm{threshold}$ is the number of the FAR values below the threshold. Note that this Monte-Carlo approximation also only works if the p(FAR) is drawn from the true distribution and not e.g. linearly.
   3. Compute the likelihood $p(\theta | \Lambda) = p(\theta)$ (the model parameters $\Lambda$ are fixed and need to be identical to the parameters used in the injections)
   4. Compute $p(\mathrm{FAR}) = \frac{N_\mathrm{detected}}{N_\mathrm{total}}$
   5. Multiply together to get $P(\mathrm{detected} | \theta) = P(\mathrm{FAR} < \mathrm{FAR_{threshold}} | \theta) = \frac{p(\theta | \mathrm{FAR} < \mathrm{FAR_{threshold}}) \cdot p(\mathrm{FAR})}{p(\theta)} = \frac{\sum_i^{N_\mathrm{threshold}}p_\mathrm{CNF}(\theta|\mathrm{FAR})}{N_\mathrm{total}\cdot p(\theta)}$ (the $N_\mathrm{threshold}$ cancels)
3. Get the detections by comparing $P(\mathrm{detected} | \theta)$ against a uniform random draw for each sample in the interval $[0,1]$ and keep only when detection probability is larger
4. Compare the distributions of parameters vs. the true underlying "detected" distribution in injections
      

p(x) is just the model for the parameters, i.e. it really is $p(x) = p(x | \Lambda)$, where $\Lambda$ describes the actual model-parameters of e.g. how the mass is distributed. First, we need to check that. The model is described (for GWTC-4) in https://arxiv.org/abs/2508.10638. I test my model for GWTC-4 in a separate notebook.

In [5]:
X_data.data_df.columns

Index(['m1_detector', 'm2_detector', 'luminosity_distance', 'a1', 'a2',
       'cos_inclination', 'sinusoidal_polarization', 't1', 't2', 'cos_phi1',
       'cos_phi2', 'sinusoidal_right_ascension', 'atan_declination'],
      dtype='str')

In [26]:
# Assumed constants
H0cosmo = 67.9 # constants from https://dcc.ligo.org/public/0170/P2000318/011/o3b_catalog.pdf
Om0cosmo = 0.3065
cosmo = FlatLambdaCDM(H0=H0cosmo * u.km / u.s / u.Mpc, Om0=Om0cosmo)
speed_of_light = constants.c.to(u.km / u.s).value

def ddL_of_z(z, dL, H0):
    return dL / (1 + z) + speed_of_light * (1 + z) / (H0 * cosmo.efunc(z))

N_eval = 1000000
mass_grid = np.linspace(1, 1000, N_eval)
spin_grid = np.linspace(0, 1, N_eval)
z_grid = np.linspace(0, 3, N_eval)
dVdz_grid = cosmo.differential_comoving_volume(z_grid).to(u.Gpc**3 / u.sr).value * 4 * np.pi

def primary_mass_pdf_interp(m1, min_m1=1.0, max_m1=1000.0):

    m = mass_grid.astype(float)
    logm = np.log(m)

    log_pdf = np.full_like(m, -np.inf)  # log(0)

    # --- piecewise definitions in log-space ---
    # [1, 3): flat
    sel = (m >= 1) & (m < 3)
    log_pdf[sel] = 0.0

    # [3, 8): (m/3)^(-4)
    sel = (m >= 3) & (m < 8)
    log_pdf[sel] = -4.0 * (logm[sel] - np.log(3.0))

    # [8, 50): continuity enforced explicitly
    sel = (m >= 8) & (m < 50)
    log_norm = -4.0 * (np.log(8.0) - np.log(3.0))
    log_pdf[sel] = log_norm - 1.0 * (logm[sel] - np.log(8.0))

    # [50, 200)
    sel = (m >= 50) & (m < 200)
    log_norm += -1.0 * (np.log(50.0) - np.log(8.0))
    log_pdf[sel] = log_norm - 4.0 * (logm[sel] - np.log(50.0))

    # [200, 1000)
    sel = (m >= 200) & (m < 1000)
    log_norm += -4.0 * (np.log(200.0) - np.log(50.0))
    log_pdf[sel] = log_norm - 1.0 * (logm[sel] - np.log(200.0))

    # Outside allowed range
    log_pdf[(m < min_m1) | (m > max_m1)] = -np.inf

    # --- exponentiate safely ---
    pdf = np.exp(log_pdf)

    # --- normalize robustly ---
    norm = np.trapezoid(pdf, m)
    if norm > 0:
        pdf /= norm

    # --- interpolate ---
    return interp1d(
        m, pdf,
        bounds_error=False,
        fill_value=0.0,
        assume_sorted=True
    )(m1)

def primary_mass_pdf(m1, min_m1=1.0, max_m1=1000.0):

    m = np.asarray(m1, dtype=float)
    logm = np.log(m, where=(m > 0), out=np.full_like(m, -np.inf))

    log_pdf = np.full_like(m, -np.inf)  # log(0)

    # --- piecewise definition in log-space ---

    # [1, 3): flat
    sel = (m >= 1) & (m < 3)
    log_pdf[sel] = 0.0

    # [3, 8): (m/3)^(-4)
    sel = (m >= 3) & (m < 8)
    log_pdf[sel] = -4.0 * (logm[sel] - np.log(3.0))

    # [8, 50)
    sel = (m >= 8) & (m < 50)
    log_norm = -4.0 * (np.log(8.0) - np.log(3.0))
    log_pdf[sel] = log_norm - (logm[sel] - np.log(8.0))

    # [50, 200)
    sel = (m >= 50) & (m < 200)
    log_norm += -1.0 * (np.log(50.0) - np.log(8.0))
    log_pdf[sel] = log_norm - 4.0 * (logm[sel] - np.log(50.0))

    # [200, 1000)
    sel = (m >= 200) & (m < 1000)
    log_norm += -4.0 * (np.log(200.0) - np.log(50.0))
    log_pdf[sel] = log_norm - (logm[sel] - np.log(200.0))

    # Outside bounds
    log_pdf[(m < min_m1) | (m > max_m1)] = -np.inf

    # --- exponentiate safely ---
    pdf = np.exp(log_pdf)

    # --- robust normalization ---
    # Sort for trapezoid stability
    idx = np.argsort(m)
    m_sorted = m[idx]
    pdf_sorted = pdf[idx]

    norm = np.trapezoid(pdf_sorted, m_sorted)
    if norm > 0:
        pdf /= norm

    return pdf


def secondary_mass_pdf(m2, m1, min_m2 = 1):

    pm2 = 2*m2 / (m1**2 - 1)
    support = (m2 >= min_m2) & (m2 <= m1)
    # should be properly normalized already I realize
    return np.where(support, pm2, 0.0)

def spin_pdf(s):

    ps = np.exp(-2*s**2)

    idx = np.argsort(s)
    s_sorted = s[idx]
    pdf_sorted = ps[idx]
    
    norm = np.trapezoid(pdf_sorted, s_sorted)
    if norm > 0:
        ps /= norm

    return ps

def cos_tilts_pdf(tilt):
    """Includes Jacobian to go from tilt to cos_tilt"""
    cos_tilt = np.cos(tilt)
    # should be normalized given the definition
    pcost = (0.3 * ((1+cos_tilt)**3)/4 + 0.35)*np.sin(tilt)  # last summand is 0.7*1/2
    return pcost

def redshift_pdf(z, z_grid, dVdz_grid):
    num_grid = dVdz_grid
    den = np.trapezoid(num_grid, z_grid)
    num_grid /= den
    
    return interp1d(z_grid, num_grid, fill_value = 0, bounds_error = False)(z)

In [75]:
def calc_p_model(data: pd.DataFrame, redshifts: np.ndarray |None = None) -> np.ndarray:
    """Calculates the p(theta) for GWTC and specific 13 parameters

    Args:
        data: DataFrame containing the parameters.

    Returns:
        Model probabilites.
    """

    if "redshift" not in data.columns and redshifts is None:
        data["redshift"] = z_at_value(cosmo.luminosity_distance, data["luminosity_distance"].to_numpy()*u.Mpc, zmax = 3).value
    elif redshifts is not None:
        data["redshift"] = redshifts
        
    data["m1_source"] = data["m1_detector"] / (1+data["redshift"])
    data["m2_source"] = data["m2_detector"] / (1+data["redshift"])

    pm1 = primary_mass_pdf(data["m1_source"])
    pm2 = secondary_mass_pdf(data["m2_source"], data["m1_source"])
    ps1 = spin_pdf(data["a1"])
    ps2 = spin_pdf(data["a2"])
    pt1 = cos_tilts_pdf(data["t1"])
    pt2 = cos_tilts_pdf(data["t2"])
    pz = redshift_pdf(data["redshift"], z_grid, dVdz_grid)

    # Make sure data is correctly drawn from the below pdfs (follows GWTC-4)

    pphi1 = np.ones_like(data["phi1"]) / (2*np.pi) # uniform in [0, 2pi]
    pphi2 = np.ones_like(data["phi2"]) / (2*np.pi) # uniform in [0, 2pi]
    pra = np.ones_like(data["right_ascension"]) / (2*np.pi) # uniform in [0, 2pi]
    ppol = np.ones_like(data["polarization"]) / np.pi # uniform in [0, pi]
    pdec = np.cos(data["declination"]) / 2 # uniform in sin(delta) [-1, 1]
    pinc = np.sin(data["inclination"]) / 2 # uniform in cos(inc) [-1, 1]

    return pm1*pm2*pa1*pa2*pz*pt1*pt2*pphi1*pphi2*pra*pdec*ppol*pinc

In [69]:
def calc_pCNF(flow: Transform, theta: np.ndarray, far: np.ndarray) -> float:
    """Calculate the detection probability.

    Args:
        flow: trained CNF.
        theta: GW parameters in the correct order for the flow.
        far: the conditioning variable (need to be drawn from the true distribution) and already cut to the FAR threshold.

    Returns:
        pCNF.
    """
    theta_reshaped = np.tile(theta, (len(far), 1)) # gives (N_threshold, dimension of theta)

    log_probs = flow.log_prob(theta_reshaped, condition=far[far_sel])
    pCNF = np.exp(logsumexp(log_probs))

    p_model = (theta)

    return pCNF

In [70]:
X_data.data_df.columns

Index(['m1_detector', 'm2_detector', 'luminosity_distance', 'a1', 'a2',
       'cos_inclination', 'sinusoidal_polarization', 't1', 't2', 'cos_phi1',
       'cos_phi2', 'sinusoidal_right_ascension', 'atan_declination',
       'inclination', 'polarization', 'phi1', 'phi2', 'right_ascension',
       'declination'],
      dtype='str')

In [76]:
def calc_p_detection(flow: Transform, data_item: ut.Data,  conditioning_item: ut.Data, redshifts: np.ndarray |None = None, far_threshold: float = 1) -> np.ndarray:

    # contrary to what the json file says, I did whiten the X data, but not the y (conditioning) data

    theta = data_item.whitened_data.copy() # make sure we work on the gw parameters only for the flow
    conditioning = conditioning_item.samples_transformed

    data = data_item.data_df

    # now add some quantities to my data
    data["inclination"] = np.arccos(data["cos_inclination"])
    data["polarization"] = (1 - np.cos(data["sinusoidal_polarization"])) * np.pi / 2 
    data["phi1"] = np.arccos(data["cos_phi1"])
    data["phi2"] = np.arccos(data["cos_phi2"])
    data["right_ascension"] = (1 - np.cos(data["sinusoidal_right_ascension"])) * np.pi
    data["declination"] = np.tan(data["atan_declination"])

    far = conditioning_item.data_df

    far_sel = far < far_threshold
    N_total = len(far)
    
    print(f"Calculating model for {np.sum(far_sel)} data...")
    p_model = calc_p_model(data[far_sel], redshifts[far_sel])

    pCNFs = []
    for data_row in tqdm(theta):
        pCNFs.append(calc_pCNF(flow, theta, conditioning[far_sel]))

    pCNFs = np.array(pCNFs)

    return pCNFs / (p_model * N_total)

    

In [99]:
# quickly read in the redshifts from the file
from pathlib import Path
fname = Path("/project/ls-gruen/users/julius.gassert/data/samples-rpo4a_v2_20250503133839UTC-1366933504-23846400.hdf")
raw_injection_data = h5py.File(fname, 'r')
events = raw_injection_data['events']
injectionData = pd.DataFrame()
injectionData['redshift'] = events["z"][()]

In [74]:
p_detections = calc_p_detection(loaded_flow, X_data, y_data)

Calculating model for 476406 data...


Precision nan reached after 3 function calls. [astropy.cosmology._src.funcs.optimize]


KeyboardInterrupt: 

In [67]:
loaded_config

{'base_dist': 'Normal',
 'data_dim': 13,
 'key': 0,
 'cond_dim': 1,
 'flow_layers': 6,
 'nn_width': 128,
 'nn_depth': 6,
 'type': 'MAF',
 'trained_columns': ['m1_detector',
  'm2_detector',
  'luminosity_distance',
  'a1',
  'a2',
  'cos_inclination',
  'sinusoidal_polarization',
  't1',
  't2',
  'cos_phi1',
  'cos_phi2',
  'sinusoidal_right_ascension',
  'atan_declination']}

In [44]:
y_data.samples_transformed

Array([[14.77407203],
       [14.77407203],
       [14.77407203],
       ...,
       [14.77407203],
       [14.77407203],
       [14.77407203]], dtype=float64)

In [45]:
y_data.data_df

,far_min
0,inf
1,inf
2,inf
3,inf
4,inf
...,...
1499239,inf
1499240,inf
1499241,inf
1499242,inf
